# Notebook 04: Backpropagation

**Module:** 05 Deep Learning  
**Duration:** ~3 hrs

---

## Learning Objectives

1. Derive backpropagation using chain rule
2. Implement backward pass in NumPy
3. Compute gradients for each layer
4. Understand why autograd exists

## 1. Derivation (Binary Cross-Entropy + Sigmoid Output)

**Loss:** $L = -[y\log\hat{y} + (1-y)\log(1-\hat{y})]$

**Output layer gradient:**
$$\frac{\partial L}{\partial Z^{[L]}} = \hat{Y} - Y = A^{[L]} - Y$$

**Hidden layer (chain rule):**
$$\frac{\partial L}{\partial Z^{[l]}} = (W^{[l+1]T} \frac{\partial L}{\partial Z^{[l+1]}}) \odot \sigma'(Z^{[l]})$$

**Weight gradient:**
$$\frac{\partial L}{\partial W^{[l]}} = \frac{1}{m} \frac{\partial L}{\partial Z^{[l]}} A^{[l-1]T}$$

## 2. ReLU derivative

$$\text{ReLU}'(z) = \begin{cases} 1 & z > 0 \\ 0 & z \leq 0 \end{cases}$$

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

REPO = Path('../../').resolve()
plt.rcParams['figure.figsize'] = (8, 5)
rng = np.random.default_rng(42)

In [ ]:
def relu_derivative(z): return (z > 0).astype(float)

def backward_pass(y_true, activations, zs, weights):
    """Backprop for BCE + sigmoid output or MSE + linear output."""
    m = y_true.shape[0]
    grads_W, grads_b = [], []
    # Output layer (sigmoid + BCE → dZ = A - Y)
    dZ = activations[-1] - y_true.reshape(-1, 1)
    for l in reversed(range(len(weights))):
        grads_W.insert(0, (dZ.T @ activations[l]) / m)
        grads_b.insert(0, dZ.mean(axis=0))
        if l > 0:
            dA = dZ @ weights[l]
            dZ = dA * relu_derivative(zs[l-1])
    return grads_W, grads_b

print('Backpropagation computes gradients layer by layer via chain rule.')

## 3. PyTorch Autograd Preview

PyTorch computes all of this automatically:
```python
loss.backward()  # runs backprop
w.grad           # gradients appear here
```

You must understand the math to debug when training fails.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset

plt.rcParams['figure.figsize'] = (8, 5)
torch.manual_seed(42)
rng = np.random.default_rng(42)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device: {device}')

In [ ]:
x = torch.tensor([[1.0, 2.0]], requires_grad=True)
w = torch.tensor([[0.5, -0.3]], requires_grad=True)
b = torch.tensor([0.1], requires_grad=True)

y = x @ w.T + b
loss = y.sum()
loss.backward()

print(f'dL/dw = {w.grad}')
print(f'dL/db = {b.grad}')

## Exercise

For a 2-layer network, compute $\partial L / \partial W^{[1]}$ by hand for one sample. Verify with PyTorch autograd.

In [ ]:
# YOUR CODE HERE


---

## Summary

Backpropagation = chain rule applied systematically through the network.

**Next:** [05_Activation_Functions.ipynb](05_Activation_Functions.ipynb)